## Synthetic Python Function Dataset Generator

This section provides Python function templates designed to generate synthetic code examples. Each function template includes random elements controlled by a seed, allowing for reproducible variations. A main function is also provided to generate a specified number of unique datasets.

# **Synthetic Data Generator**

In [ ]:
import json
import random
import re
from pathlib import Path

CONFIG = {
    "seed": 42,
    "n_train": 1000,
    "n_eval": 100,
    "out_dir": "math_reasoning_dataset",  # Changed from Colab path to relative path
    "train_min_difficulty": 1,
    "train_max_difficulty": 7,
    "eval_min_difficulty": 5,
    "eval_max_difficulty": 10,
}

# **Installing the dependencies like vLLM and UnSloth**

In [ ]:
!pip install unsloth vllm

In [ ]:
import vllm
print(vllm.__version__)

## Load Qwen 0.5B Instruct Model using Unsloth

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.", category=FutureWarning)
warnings.filterwarnings("ignore", message="Both `max_new_tokens`.*and `max_length`.*seem to have been set.*", category=UserWarning)
warnings.filterwarnings("ignore", message="The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.", category=FutureWarning)

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
print('CUDA_LAUNCH_BLOCKING set to 1 for debugging.')

In [ ]:
import torch
seed = 42
torch.manual_seed(seed)
print(f"Random seed set to {seed}")

# **GRPO Training**

In [ ]:
!uv pip install https://github.com/vllm-project/vllm/releases/download/v0.23.0/vllm-0.23.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl

In [ ]:
from transformers import AutoProcessor
MODEL_ID = "unsloth/Qwen2.5-0.5B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
import torch
import re
import wandb
import matplotlib.pyplot as plt
# from datasets import load_dataset
from unsloth import FastLanguageModel

**defining model id**

In [ ]:
MODEL_ID = "unsloth/Qwen2.5-0.5B-Instruct"

# **soft launching the model**

In [ ]:
max_seq_length = 850
dtype = None

lora_rank = 32
lora_alpha = 16
lora_dropout = 0.05

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = True,
    max_lora_rank = lora_rank,
    fast_inference = True,
    gpu_memory_utilization = 0.4,   # "i have pretty low vram so keeping it to 0.4 which gave pretty good memorymanagement"
)

**adding LoRA adapter over qko layer**

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
     target_modules = [
         "q_proj", "k_proj", "o_proj"
         ],
    lora_alpha = 16,
    lora_dropout = 0,             # unsloth is optimized for dropout=0
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # big activation-memory saver
    random_state = 3407,
)

model.print_trainable_parameters()  # sanity check: you want a small % here, not 0% and not 100%

**setting padding location to left for the grpo training**

In [ ]:
tokenizer.padding_side = "left"
if tokenizer.pad_token_id == tokenizer.eos_token_id:
    tokenizer.add_special_tokens({
        "pad_token": "<|pad|>"
    })
    print("Pad_token_id is created!!")

model.resize_token_embeddings(len(tokenizer))

model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

print("PAD:", tokenizer.pad_token_id)
print("EOS:", tokenizer.eos_token_id)

# **datatset**

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(train)
eval_dataset = Dataset.from_list(eval_)
eval_paraphrased_dataset = Dataset.from_list(eval_paraphrased)

print(train_dataset)

In [ ]:
train_dataset = train_dataset.rename_column("prompt", "question")
train_dataset = train_dataset.rename_column("completion", "answer")

eval_dataset = eval_dataset.rename_column("prompt", "question")
eval_dataset = eval_dataset.rename_column("completion", "answer")
eval_paraphrased_dataset = eval_paraphrased_dataset.rename_column("completion", "answer")

print(train_dataset)
print(eval_dataset)

**sanity check:)**

In [ ]:
print(train_dataset['question'][10])
print(train_dataset['answer'][10])

In [ ]:
import re

# Create a proper clone of eval_dataset before mutations
clone_eval_dataset = eval_dataset.select(range(len(eval_dataset)))

def format_answer(ds):
  ans = ds['answer']
  messages1 = [
      {
          "role": "user",
          "content": ds['question'],
      },
  ]

  ds['prompt'] = messages1
  ds['solution'] = float(ans)
  return ds

# Apply map
train_dataset = train_dataset.map(format_answer, batched=False)
valid_dataset = eval_dataset.map(format_answer, batched=False)
eval_paraphrased_dataset = eval_paraphrased_dataset.map(format_answer, batched=False)

# Filter out entries where 'prompt' or 'solution' couldn't be generated
train_dataset = train_dataset.filter(lambda x: 'prompt' in x and 'solution' in x)
valid_dataset = valid_dataset.filter(lambda x: 'prompt' in x and 'solution' in x)
eval_paraphrased_dataset = eval_paraphrased_dataset.filter(lambda x: 'prompt' in x and 'solution' in x)

# Remove the original 'answer' and 'question' columns if they are not needed by the trainer
train_dataset = train_dataset.remove_columns([col for col in ['question', 'answer'] if col in train_dataset.column_names])
valid_dataset = valid_dataset.remove_columns([col for col in ['question', 'answer'] if col in valid_dataset.column_names])
eval_paraphrased_dataset = eval_paraphrased_dataset.remove_columns([col for col in ['question', 'answer'] if col in eval_paraphrased_dataset.column_names])

In [ ]:
wandb.login(key="USE YOUR OWN API KEY")

In [ ]:
def math_reward(completions, solution, **kwargs):
    rewards = []
    for completion, sol in zip(completions, solution):
        # unwrap conversational format: [{"role": "assistant", "content": "..."}] -> "..."
        if isinstance(completion, list):
            text = completion[0]["content"]
        else:
            text = completion   # already a plain string (non-conversational prompt format)
        nums = re.findall(r"-?\d+(?:\.\d+)?", text)

        if nums is not None:
            try:
                model_answer = nums[-3:]
                model_answer = [float(n) for n in model_answer]
                rewards.append(1.0 if float(sol) in model_answer else 0.0)
            except ValueError:
                rewards.append(0.0)
        else:
            rewards.append(0.0)
    return rewards

In [ ]:
from trl import GRPOTrainer, GRPOConfig

In [ ]:
max_seq_length = 1024

In [ ]:
training_args = GRPOConfig(
    output_dir = "qwen0.5b-math-grpo",
    learning_rate = 5e-5,

    use_vllm = True,          # required alongside fast_inference=True above — this wires TRL to Unsloth's vLLM engine
    num_generations = 8,      # start here, raise toward 32 once this runs clean — see note  below

    max_prompt_length = 256,
    max_completion_length = max_seq_length - 256,

    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,

    optim = "adamw_8bit",
    num_train_epochs = 1,
    logging_steps = 1,
    save_steps = 100,
    report_to = "wandb",

    beta = 0.01,   # KL off by default — matches TRL's own default, set >0 to turn on ref-model KL

     # --- new: evaluation ---
    eval_strategy = "no",   # I didnt passed because I was having the GPU constraint.
    # eval_steps = 100,
    # eval_batch_size = 4,
)

In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,   # Unsloth's model/tokenizer, not a raw HF model
    reward_funcs = math_reward,
    args = training_args,
    train_dataset = train_dataset, # Changed from 'dataset' to 'train_dataset'
    eval_dataset = valid_dataset,  # Changed from 'dataset' to 'eval_dataset'
)

In [ ]:
trainer.train()      

In [ ]:
model.save_pretrained("my-math-grpo-lora")
tokenizer.save_pretrained("my-math-grpo-lora")

In [ ]:
import shutil

shutil.make_archive(
    "my-math-grpo-lora",
    "zip",
    "my-math-grpo-lora"
)

In [ ]:
# Updated math_reward function for evaluation (handles batched completions properly)
def math_reward(completions, solution, **kwargs):
    rewards = []
    # The evaluation uses num_return_sequences = 4
    num_generations = kwargs.get("num_generations", 4) 

    # `completions` is a flat list of `batch_size * num_generations` generated texts.
    # `solution` is a list/tensor of `batch_size` actual solution values (strings).

    batch_size = len(solution)

    for i in range(batch_size):
        # Extract the true solution for the current prompt in the batch
        true_sol = float(solution[i])

        # Extract the `num_generations` completions for the current prompt
        prompt_completions = completions[i * num_generations : (i + 1) * num_generations]

        for completion_text in prompt_completions:
            # unwrap conversational format if needed
            if isinstance(completion_text, list): # Check if it's a list of dicts (messages)
                text_to_parse = completion_text[0]["content"] if completion_text else ""
            else: # Otherwise, it's already a string
                text_to_parse = completion_text

            nums = re.findall(r"-?\d+(?:\.\d+)?", text_to_parse)
            reward = 0.0
            if nums:
                try:
                    # Model's answer is typically the last number, or last few
                    model_answer_candidates = [float(n) for n in nums[-3:]] # Assuming last 3 numbers are potential answers
                    if true_sol in model_answer_candidates:
                        reward = 1.0
                except ValueError:
                    pass # Keep reward as 0.0 if conversion fails
            rewards.append(reward)
    return rewards

# **TESTING ON VALID_DATASET**

In [ ]:
valid_dataset['solution']

In [ ]:
import torch

all_index = torch.arange(len(valid_dataset))
all_index

In [ ]:
import numpy as np
import time

def evaluation(valid_dataset, file_name):

    dicts = {}
    batch_size = 10
    num_return_sequences = 4

    for start in range(0, len(valid_dataset), batch_size):
        end = min(start + batch_size, len(valid_dataset))
        idxes = list(range(start, end))   

        batch_prompts_raw = [valid_dataset['prompt'][i] for i in idxes]
        batch_solutions    = [valid_dataset['solution'][i] for i in idxes]

        prompts = [
            tokenizer.apply_chat_template(p, tokenize=False, add_generation_prompt=True)
            for p in batch_prompts_raw
        ]

        toks = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True).to("cuda")
        ids = toks['input_ids']
        msk = toks['attention_mask']

        s = time.time()
        o_ids = model.generate(
            input_ids=ids,
            attention_mask=msk,
            temperature=0.6,
            top_p=0.95,
            do_sample=True,
            max_new_tokens=850,
            num_return_sequences=num_return_sequences,
        )
        e = time.time()

        prompt_len = ids.shape[1]
        gen_only_ids = o_ids[:, prompt_len:]
        all_responses = tokenizer.batch_decode(gen_only_ids, skip_special_tokens=True)

        for j, orig_idx in enumerate(idxes):
            block_start = j * num_return_sequences
            block_end = block_start + num_return_sequences
            responses_for_this_example = all_responses[block_start:block_end]
            sln = batch_solutions[j]

            rewards = math_reward(responses_for_this_example, [sln] * num_return_sequences)
            correct = int(np.sum(np.array(rewards)))
            
            # Fixed: calculate mean and std from rewards array, not from integer
            dicts[orig_idx] = {
                'rewards': rewards, 
                'correct': correct, 
                'mean': np.mean(rewards), 
                'std': np.std(rewards)
            }
            print(f' === Idx: {orig_idx} | Correct: {correct} / {num_return_sequences} ===')

        print(f'--- Batch [{start}:{end}] done in {(e - s):.2f}s ---')

    import json

    with open(f'{file_name}.json', 'w') as f: 
        json.dump(dicts, f, indent=4)
    print(f'{file_name}.json saved successfully.')

# **normal prompting**

evaluation(valid_dataset, 'normal_prompting')

# **CoT Promting**

In [ ]:
# **CoT Prompting**
# Use the cloned dataset which still has original columns
def format_answer_cot(ds):
  ans = ds['answer']
  messages1 = [
      {
          "role": "user",
          "content": f"{ds['question']}\n\n"
                "Solve the problem step by step. Explain the reasoning clearly, "
                "show the necessary intermediate calculations, and provide the "
                "final answer at the end.",
      },
  ]

  ds['prompt'] = messages1
  ds['solution'] = float(ans)
  return ds

# Apply map to the cloned dataset which still has 'question' and 'answer'
cot_valid_dataset = clone_eval_dataset.map(format_answer_cot, batched=False)

# Filter out entries where 'prompt' or 'solution' couldn't be generated
cot_valid_dataset = cot_valid_dataset.filter(lambda x: 'prompt' in x and 'solution' in x)

# Remove the original 'answer' and 'question' columns
cot_valid_dataset = cot_valid_dataset.remove_columns([col for col in ['question', 'answer'] if col in cot_valid_dataset.column_names])

In [ ]:
evaluation(cot_valid_dataset, 'cot_prompting')

# **GRPO Adapter**

In [ ]:
# **GRPO Adapter**
# Attach trained LoRA adapter
from peft import PeftModel

ADAPTER_PATH = "my-math-grpo-lora"

model = PeftModel.from_pretrained(
    model,
    ADAPTER_PATH,
)

model.set_adapter("default")
_ = model.eval()

In [ ]:
evaluation(valid_dataset, 'grpo_fine_tuned')
evaluation(eval_paraphrased_dataset, 'grpo_fine_tuned_reparaphrased')